# Python Bridge for 11b

## A Short Code Refresher

Use this optional notebook alongside 11b to practice aligning earlier observations, calculating changes, and reading autocorrelation results. Allow about 15–20 minutes. A small quarterly series supplies every value needed here.

The central Python distinction is between a date label and a row offset. Once the calendar is checked, a shift of four rows means the same quarter one year earlier.

## References

Pandas documents [Series.shift](https://pandas.pydata.org/docs/reference/api/pandas.Series.shift.html) and [Series.diff](https://pandas.pydata.org/docs/reference/api/pandas.Series.diff.html). The statsmodels references for [acf](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.acf.html) and [acorr_ljungbox](https://www.statsmodels.org/stable/generated/statsmodels.stats.diagnostic.acorr_ljungbox.html) describe their returned arrays and tables. The [course errata](https://auburn.instructure.com/courses/1747056/pages/course-addenda-and-errata#11b-the-adjusted-option-in-autocorrelation) explains the ACF normalization used in the lecture.

## Create a Quarterly Series

The sixteen invented observations contain a repeated quarterly pattern and an increase of one unit per year. A `PeriodIndex` labels quarters as intervals rather than particular days.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import acf

quarters = pd.period_range("2020Q1", periods=16, freq="Q")
series = pd.Series(
    [10, 15, 12, 8, 11, 16, 13, 9, 12, 17, 14, 10, 13, 18, 15, 11],
    index=quarters,
    name="Observed",
)
assert series.index.is_unique
assert series.index.equals(pd.period_range(series.index.min(), series.index.max(), freq="Q"))
display(series.head())

The complete quarterly calendar is important: a row offset represents elapsed quarters here because no quarter is missing.

## Align an Earlier Observation

`.shift(1)` keeps the index labels and moves the values down one row. Put the original and shifted Series in a DataFrame to inspect the actual pairs.

In [ ]:
lagged = pd.DataFrame({"Current": series, "Previous quarter": series.shift(1)})
display(lagged.head())
pairs = lagged.dropna()
print("Complete lag-one pairs:", len(pairs))

At 2020Q2, the pair is current 15 and previous-quarter 10. The first row lacks a preceding observation, leaving 15 complete pairs. No `freq` argument was supplied to `shift`: we moved values by row position rather than moving the date labels.

## Calculate Ordinary and Seasonal Differences

`.diff(1)` subtracts the previous row. `.diff(4)` subtracts the row four positions earlier. Display both beside their source observations before dropping unavailable results.

In [ ]:
differences = pd.DataFrame(
    {
        "Current": series,
        "Previous quarter": series.shift(1),
        "Quarterly change": series.diff(),
        "Same quarter last year": series.shift(4),
        "Annual change": series.diff(4),
    }
)
display(differences.head(7))
annual_change = differences["Annual change"].dropna()
assert np.allclose(series.diff(4).dropna(), (series - series.shift(4)).dropna())

At 2021Q1, ordinary differencing gives 11 − 8 = 3, while seasonal differencing gives 11 − 10 = 1. The annual changes are all one because that increase was built into this example. Four initial seasonal differences are unavailable. `.diff(4)` is one lag-four subtraction, not four repeated first differences.

## Read the ACF Array

`acf` returns one coefficient per lag, including lag zero. Request four lags and attach explicit lag numbers so the array positions remain clear. `adjusted=False` matches the lecture's full-record denominator; `fft=False` requests direct computation.

In [ ]:
coefficients = acf(series, nlags=4, adjusted=False, fft=False)
acf_table = pd.Series(coefficients, index=range(len(coefficients)), name="ACF")
acf_table.index.name = "Lag (quarters)"
display(acf_table.round(3))
print("Returned shape:", coefficients.shape)

There are five entries, with lag zero equal to one. Lag four compares corresponding quarters across years. These coefficients use the ACF's overall-mean calculation; they need not equal a separate Pearson correlation of each shortened pair of columns.

Plot the levels and ordinary changes to see the quantities whose dependence we might examine.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True, layout="constrained")
plot_dates = series.index.to_timestamp()
axes[0].plot(plot_dates, series, "o-")
axes[1].plot(plot_dates, series.diff(), "o-")
axes[0].set(title="Constructed Quarterly Series", ylabel="Level (units)")
axes[1].set(ylabel="Quarterly change (units)", xlabel="Quarter start")
plt.show()

Differencing changes the measured quantity. It does not automatically remove all temporal structure; the quarterly change plot still repeats a pattern.

## Read a Joint-Test Result Table

`lags=[4]` requests one Ljung–Box result testing zero autocorrelation jointly across lags one through four. It does not request a test of lag four alone. The returned DataFrame uses the tested maximum lag as its index.

In [ ]:
joint_test = acorr_ljungbox(series, lags=[4], return_df=True)
display(joint_test)
print("P-value for the lags 1–4 test:", joint_test.loc[4, "lb_pvalue"])

`lb_stat` is the test statistic and `lb_pvalue` is its p-value. This tiny constructed series is for reading the interface; it is not a suitable basis for claiming that a real process is white noise or stationary. Neither a large p-value nor a quiet sample ACF proves independence.

## Try One Small Modification

Drop one quarter before shifting. At 2021Q1, will a shift of four rows still retrieve 2020Q1? Restore the complete calendar and compare.

In [ ]:
with_gap = series.drop(pd.Period("2020Q2", freq="Q"))
calendar = pd.period_range(series.index.min(), series.index.max(), freq="Q")
restored = with_gap.reindex(calendar)
display(pd.DataFrame({"With gap": with_gap, "Four rows earlier": with_gap.shift(4)}).head(6))
display(pd.DataFrame({"Restored": restored, "Four quarters earlier": restored.shift(4)}).head(6))

In the shortened table, 2021Q1 is only the fourth row, so it has no lag-four partner. Restoring the calendar retains the missing quarter as `NaN` and puts 2020Q1 four rows before 2021Q1 again. Restoring dates does not fill the missing observation.

## Ready for 11b

Return to the lecture when you can inspect lag pairs, distinguish ordinary and seasonal differences, label the lag-zero entry in an ACF array, and read a joint-test result by row and column name.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the [INSY 7130 course-materials README](../../README.md).